# pdf_vlm — Document QA experiment (Colab)

## What this experiment measures

For Hyundai WIA report packs at **5 / 10 / 20 / 50 / 100 pages**:

| Axis | Variants |
|---|---|
| OCR | PP-StructureV3 (**tables ON** on Colab) + PDF text enrich |
| RAG generation | **text-only** vs **multimodal** (page images) |
| Retrieval | **page-level** vs **hierarchical** |
| Metric | ANLS / EM / F1 + recall@k (needs Gemma GGUF for real answers) |

**Runtime:** GPU (T4+) recommended.

> Do **not** clone into `/content/pdf_vlm` — that folder name shadows the Python package.

## 0. Clone + install (package + OCR + llama.cpp)

In [1]:
import sys, shutil
from pathlib import Path

REPO_URL = "https://github.com/mAn-He/pdf_vlm.git"
ROOT = Path("/content/pdf_vlm_repo")

# Remove shadowed clone path if present
shadow = Path("/content/pdf_vlm")
if shadow.exists() and shadow.resolve() != ROOT.resolve():
    shutil.rmtree(shadow, ignore_errors=True)

if not (ROOT / "pyproject.toml").exists():
    !git clone --depth 1 {REPO_URL} {ROOT}
else:
    print("Repo present:", ROOT)

%cd {ROOT}
for k in list(sys.modules):
    if k == "pdf_vlm" or k.startswith("pdf_vlm."):
        del sys.modules[k]
sys.path.insert(0, str(ROOT / "src"))
print("cwd:", Path.cwd())

Cloning into '/content/pdf_vlm_repo'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (213/213), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 213 (delta 32), reused 194 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (213/213), 1.18 MiB | 16.79 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/pdf_vlm_repo
cwd: /content/pdf_vlm_repo


In [2]:
import subprocess, sys
from pathlib import Path

ROOT = Path("/content/pdf_vlm_repo").resolve()
assert (ROOT / "src/pdf_vlm/utils/io.py").exists()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
# index + OCR (tables) + viz
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[index,ocr,viz]"])

def try_install_llama():
    for url in [
        "https://abetlen.github.io/llama-cpp-python/whl/cu124",
        "https://abetlen.github.io/llama-cpp-python/whl/cu122",
        None,
    ]:
        cmd = [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"]
        if url:
            cmd += ["--extra-index-url", url]
        print("Trying llama-cpp:", url or "default")
        if subprocess.run(cmd).returncode == 0:
            return True
    return False

print("llama-cpp:", try_install_llama())

for k in list(sys.modules):
    if k == "pdf_vlm" or k.startswith("pdf_vlm."):
        del sys.modules[k]
sys.path.insert(0, str(ROOT / "src"))

import pdf_vlm
from pdf_vlm.utils.io import project_root
from pdf_vlm.ocr.paddle_structure import paddle_available
print("pdf_vlm:", pdf_vlm.__file__)
print("paddle:", paddle_available())

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Trying llama-cpp: https://abetlen.github.io/llama-cpp-python/whl/cu124
Trying llama-cpp: https://abetlen.github.io/llama-cpp-python/whl/cu124
llama-cpp: True
llama-cpp: True
pdf_vlm: /content/pdf_vlm_repo/src/pdf_vlm/__init__.py
pdf_vlm: /content/pdf_vlm_repo/src/pdf_vlm/__init__.py
paddle: {'paddleocr': True, 'PPStructureV3': True, 'PPStructure': False, 'PaddleOCR': True}
cuda: True Tesla T4
paddle: {'paddleocr': True, 'PPStructureV3': True, 'PPStructure': False, 'PaddleOCR': True}
cuda: True Tesla T4


## 1. Download Gemma GGUF (required for real QA answers)

1. Accept: https://huggingface.co/google/gemma-3-4b-it-qat-q4_0-gguf  
2. Colab secret `HF_TOKEN` or paste token  
3. Set `DOWNLOAD_GGUF = True` below

If GGUF is missing, the harness can still measure **retrieval**, but **ANLS/QA quality will be empty** (dry-run).

In [3]:
from pathlib import Path
import os

DOWNLOAD_GGUF = True  # set False only if you intentionally skip generation

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
mmproj = Path("models/mmproj-model-f16-4B.gguf")

if DOWNLOAD_GGUF and not (gguf.exists() and mmproj.exists()):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("HF token: ")
    !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
    !{sys.executable} scripts/download_models.py --with-mmproj

print("gguf:", gguf.exists(), gguf)
print("mmproj:", mmproj.exists(), mmproj)
HAS_GGUF = gguf.exists() and mmproj.exists()
print("HAS_GGUF:", HAS_GGUF)

HF token: ··········

Hint: A new version of huggingface_hub (1.24.0) is available! You are using version 1.23.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help


Hint: A new version of huggingface_hub (1.24.0) is available! You are using version 1.23.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

[15:07:29] INFO pdf_vlm: Downloading google/gemma-3-4b-it-qat-q4_0-gguf / gemma-3-4b-it-q4_0.gguf
[15:07:29] INFO pdf_vlm: Downloading google/gemma-3-4b-it-qat-q4_0-gguf / gemma

## 2. Build length packs 5/10/20/50/100 (optional if already in repo)

Repo already ships truncated PDFs under `data/custom/{5,10,20,50,100}/`.  
If you uploaded full `QA_report_HW.pdf` to the repo root, you can rebuild packs here.

In [4]:
from pathlib import Path
import sys

BUCKETS = "5,10,20,50,100"
src = Path("QA_report_HW.pdf")

if src.exists():
    !{sys.executable} scripts/prepare_hw_report_dataset.py --pdf {src} --buckets {BUCKETS}
else:
    print("No QA_report_HW.pdf in repo root — using existing data/custom packs.")

for b in [5, 10, 20, 50, 100]:
    d = Path(f"data/custom/{b}")
    pdfs = list(d.glob("*.pdf")) if d.exists() else []
    print(f"bucket={b}: pdfs={[p.name for p in pdfs]} q={(d/'questions.json').exists()}")

No QA_report_HW.pdf in repo root — using existing data/custom packs.
bucket=5: pdfs=['acme_demo_5pages.pdf', 'hyundai_wia_qa_report_5p.pdf'] q=True
bucket=10: pdfs=['hyundai_wia_qa_report_10p.pdf'] q=True
bucket=20: pdfs=['hyundai_wia_qa_report_20p.pdf'] q=True
bucket=50: pdfs=['hyundai_wia_qa_report_50p.pdf'] q=True
bucket=100: pdfs=['hyundai_wia_qa_report_100p.pdf'] q=True


## 3. OCR (PP-StructureV3, tables ON) + build retrieval indexes

Uses `configs/ocr/pp_structure_v3_colab.yaml` (`use_table_recognition: true`).  
This is the step that actually pulls OCR/table text used by RAG.

In [5]:
import sys

BUCKETS = "5,10,20,50,100"
# Start smaller if Colab RAM is tight: BUCKETS = "5,10,20"

!{sys.executable} scripts/colab_prepare_custom.py \
  --buckets {BUCKETS} \
  --no-stub \
  --enrich-pdf-text \
  --hash-embedder \
  --ocr-config ocr/pp_structure_v3_colab.yaml \
  --force

from pdf_vlm.utils.io import load_json, resolve_path
prep = load_json(resolve_path("data/custom/colab_prepared.json"))
print(prep)
assert prep.get("items"), "No docs prepared — check data/custom manifests/PDFs"

[15:08:03] INFO pdf_vlm: paddle_available={'paddleocr': True, 'PPStructureV3': True, 'PPStructure': False, 'PaddleOCR': True} table=True
[15:08:03] INFO pdf_vlm: paddle_available={'paddleocr': True, 'PPStructureV3': True, 'PPStructure': False, 'PaddleOCR': True} table=True
[15:08:04] INFO pdf_vlm.pdf.render: Rendered 5 pages from hyundai_wia_qa_report_5p.pdf
[15:08:04] INFO pdf_vlm.pdf.render: Rendered 5 pages from hyundai_wia_qa_report_5p.pdf
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from:

## 4. Run eval matrix (RAG variants × page lengths)

Cells: **text/multimodal × page/hierarchical × custom_{5,10,20,50,100}**  
If `HAS_GGUF` is False → forced dry-run (retrieval only, ANLS≈0).

In [6]:
import subprocess, sys
from pathlib import Path

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
mmproj = Path("models/mmproj-model-f16-4B.gguf")
HAS_GGUF = gguf.exists() and mmproj.exists()

cmd = [
    sys.executable, "scripts/run_eval_harness.py",
    "--config", "configs/experiments/eval_hw_wia_colab.yaml",
    "--datasets", "custom_5,custom_10,custom_20,custom_50,custom_100",
    "--pipelines", "text,multimodal",
    "--retrievals", "page,hierarchical",
    "--top-k", "3",
    "--device", "cuda",
]
if not HAS_GGUF:
    cmd.append("--dry-run")
    print("WARNING: no GGUF → dry-run (retrieval only). Re-run section 1.")
else:
    print("GGUF found → full QA generation")

print(" ".join(cmd))
subprocess.check_call(cmd)

GGUF found → full QA generation
/usr/bin/python3 scripts/run_eval_harness.py --config configs/experiments/eval_hw_wia_colab.yaml --datasets custom_5,custom_10,custom_20,custom_50,custom_100 --pipelines text,multimodal --retrievals page,hierarchical --top-k 3 --device cuda


0

In [7]:
from pathlib import Path
import json

runs = sorted(Path("results/runs").glob("eval_*"), key=lambda p: p.stat().st_mtime, reverse=True)
print("latest:", runs[0] if runs else None)
if runs:
    for name in ["summary.json", "report.md", "aggregates.json"]:
        p = runs[0] / name
        if p.exists():
            print("====", name, "====")
            txt = p.read_text(encoding="utf-8")
            print(txt[:5000])
            break

latest: results/runs/eval_eval_hw_wia_colab_20260725_160252
==== summary.json ====
{
  "run_id": "eval_eval_hw_wia_colab_20260725_160252",
  "n_rows": 0,
  "n_skipped": 20,
  "paths": {
    "predictions_csv": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260725_160252/predictions.csv",
    "predictions_json": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260725_160252/predictions.json",
    "aggregates_json": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260725_160252/aggregates.json",
    "table_overall": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260725_160252/tables/overall.csv",
    "table_by_pipeline": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260725_160252/tables/by_pipeline.csv",
    "table_by_retrieval": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260725_160252/tables/by_retrieval.csv",
    "table_by_cell": "/content/pdf_vlm_repo/results/runs/eval_eval_hw_wia_colab_20260725_160252

## 5. Inference practicality bench (needs GGUF)

This cell intentionally skips if weights are missing — it is **not** the QA harness.

In [8]:
import sys
from pathlib import Path

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
if gguf.exists():
    !{sys.executable} scripts/bench_gemma_inference.py --repeats 2
else:
    print("Skip bench: models/gemma-3-4b-it-q4_0.gguf missing.")
    print("Fix: set DOWNLOAD_GGUF=True in section 1 and re-run that cell.")

[16:02:56] INFO pdf_vlm.llm.gemma: Using MTMDChatHandler with mmproj=/content/pdf_vlm_repo/models/mmproj-model-f16-4B.gguf
[16:02:56] INFO pdf_vlm.llm.gemma: Loading Gemma 3 with vision (mmproj=mmproj-model-f16-4B.gguf)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
add_text: <start_of_turn>user
Say OK<end_of_turn>
<start_of_turn>model

[16:02:58] INFO pdf_vlm.bench.inference: Model ready (first-call includes load) first_call_ms=2378.7 rss=2122.8
[16:02:59] INFO pdf_vlm.bench.inference: warmup 1/1
add_text: <start_of_turn>user
You are a document QA assistant. Use ONLY the evidence below.

=== EVIDENCE (top-1 of 5-page doc) ===
[Page 0 of 5] Acme Corp section 0. Founded in 1998. Product VisionX-4. Revenue context for retrieval unit 0. lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lorem lo

In [9]:
from pathlib import Path
import shutil

run = sorted(Path("results/runs").glob("eval_*"), key=lambda p: p.stat().st_mtime)[-1]
out = Path("results/colab_export")
out.mkdir(parents=True, exist_ok=True)

# 요약만 복사 (용량 작게)
for name in ["summary.json", "report.md", "aggregates.json", "config_snapshot.json", "rows.csv"]:
    src = run / name
    if src.exists():
        shutil.copy2(src, out / name)

shutil.make_archive("colab_results", "zip", out)
print("download:", Path("colab_results.zip").resolve())

download: /content/pdf_vlm_repo/colab_results.zip


In [10]:
from pathlib import Path
print(list(Path("indices").glob("*/*/manifest.json"))[:20])


[PosixPath('indices/hyundai_wia_qa_report_5p_b413d5d7a1/page_text/manifest.json'), PosixPath('indices/hyundai_wia_qa_report_5p_b413d5d7a1/hier_text/manifest.json'), PosixPath('indices/hyundai_wia_qa_report_20p_0b67cfac3d/page_text/manifest.json'), PosixPath('indices/hyundai_wia_qa_report_20p_0b67cfac3d/hier_text/manifest.json'), PosixPath('indices/hyundai_wia_qa_report_100p_6ec8997f40/page_text/manifest.json'), PosixPath('indices/hyundai_wia_qa_report_100p_6ec8997f40/hier_text/manifest.json'), PosixPath('indices/hyundai_wia_qa_report_50p_8e9a5d984d/page_text/manifest.json'), PosixPath('indices/hyundai_wia_qa_report_50p_8e9a5d984d/hier_text/manifest.json'), PosixPath('indices/hyundai_wia_qa_report_10p_e6a3b05064/page_text/manifest.json'), PosixPath('indices/hyundai_wia_qa_report_10p_e6a3b05064/hier_text/manifest.json')]


In [11]:
import sys
from pathlib import Path
import json

# stem -> 실제 index doc_id
id_map = {}
for d in Path("indices").iterdir():
    if (d / "page_text").exists():
        stem = d.name.rsplit("_", 1)[0]
        id_map[stem] = d.name
print("id_map:", id_map)

for n in [5, 10, 20, 50, 100]:
    qpath = Path(f"data/custom/{n}/questions.json")
    if not qpath.exists():
        continue
    rows = json.loads(qpath.read_text(encoding="utf-8"))
    key = f"hyundai_wia_qa_report_{n}p"
    new_id = id_map.get(key)
    if not new_id:
        print("no index for", key)
        continue
    for r in rows:
        r["doc_id"] = new_id
    qpath.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")

    mpath = Path(f"data/custom/{n}/manifest.json")
    if mpath.exists():
        man = json.loads(mpath.read_text(encoding="utf-8"))
        for d in man.get("documents") or []:
            d["doc_id"] = new_id
        mpath.write_text(json.dumps(man, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"bucket {n} -> {new_id} ({len(rows)} qs)")

id_map: {'hyundai_wia_qa_report_5p': 'hyundai_wia_qa_report_5p_b413d5d7a1', 'hyundai_wia_qa_report_20p': 'hyundai_wia_qa_report_20p_0b67cfac3d', 'hyundai_wia_qa_report_100p': 'hyundai_wia_qa_report_100p_6ec8997f40', 'hyundai_wia_qa_report_50p': 'hyundai_wia_qa_report_50p_8e9a5d984d', 'hyundai_wia_qa_report_10p': 'hyundai_wia_qa_report_10p_e6a3b05064'}
bucket 5 -> hyundai_wia_qa_report_5p_b413d5d7a1 (4 qs)
bucket 10 -> hyundai_wia_qa_report_10p_e6a3b05064 (4 qs)
bucket 20 -> hyundai_wia_qa_report_20p_0b67cfac3d (8 qs)
bucket 50 -> hyundai_wia_qa_report_50p_8e9a5d984d (11 qs)
bucket 100 -> hyundai_wia_qa_report_100p_6ec8997f40 (11 qs)


In [12]:
import sys
from pathlib import Path
has = Path("models/gemma-3-4b-it-q4_0.gguf").exists()
cmd = [
  sys.executable, "scripts/run_eval_harness.py",
  "--config", "configs/experiments/eval_hw_wia_colab.yaml",
  "--datasets", "custom_5,custom_10,custom_20,custom_50,custom_100",
  "--pipelines", "text,multimodal",
  "--retrievals", "page,hierarchical",
]
if not has:
    cmd.append("--dry-run")
print(" ".join(cmd))
import subprocess; subprocess.check_call(cmd)

/usr/bin/python3 scripts/run_eval_harness.py --config configs/experiments/eval_hw_wia_colab.yaml --datasets custom_5,custom_10,custom_20,custom_50,custom_100 --pipelines text,multimodal --retrievals page,hierarchical


CalledProcessError: Command '['/usr/bin/python3', 'scripts/run_eval_harness.py', '--config', 'configs/experiments/eval_hw_wia_colab.yaml', '--datasets', 'custom_5,custom_10,custom_20,custom_50,custom_100', '--pipelines', 'text,multimodal', '--retrievals', 'page,hierarchical']' returned non-zero exit status 1.

In [13]:
import subprocess, sys
from pathlib import Path

cmd = [
    sys.executable, "scripts/run_eval_harness.py",
    "--config", "configs/experiments/eval_hw_wia_colab.yaml",
    "--datasets", "custom_5",
    "--pipelines", "text",
    "--retrievals", "page",
    "--top-k", "3",
    "--limit", "2",
    "--dry-run",   # 먼저 검색만
]
print(" ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print("exit:", r.returncode)
print("===== STDOUT =====")
print(r.stdout[-8000:] if r.stdout else "")
print("===== STDERR =====")
print(r.stderr[-8000:] if r.stderr else "")

/usr/bin/python3 scripts/run_eval_harness.py --config configs/experiments/eval_hw_wia_colab.yaml --datasets custom_5 --pipelines text --retrievals page --top-k 3 --limit 2 --dry-run
exit: 0
===== STDOUT =====
[16:47:07] INFO pdf_vlm.index.text_embedder: Loaded text embedder BAAI/bge-m3 on cuda
[16:47:08] INFO pdf_vlm.rag.text_only: qa_id=hw5_q_company retrieval=page pages=[3, 0, 4] latency_ms=ret:14033.2 gen:0.0 e2e:14033.2 answer=''
[16:47:09] INFO pdf_vlm.rag.text_only: qa_id=hw5_q_overview retrieval=page pages=[4, 0, 1] latency_ms=ret:706.9 gen:0.0 e2e:706.9 answer=''
[16:47:09] INFO pdf_vlm.eval.harness: cell custom_5__text_page_k3 -> 2 rows (correctness_mean=0.0000)
[16:47:09] INFO pdf_vlm.eval.viz: Wrote figure /content/pdf_vlm_repo/results/figures/eval_eval_hw_wia_colab_20260725_164652/length_vs_accuracy.png
[16:47:09] INFO pdf_vlm.eval.viz: Wrote figure /content/pdf_vlm_repo/results/figures/eval_eval_hw_wia_colab_20260725_164652/pipeline_latency.png
[16:47:10] INFO pdf_vlm.eval

In [14]:
import json
from pathlib import Path

q = json.loads(Path("data/custom/5/questions.json").read_text(encoding="utf-8"))
doc = q[0]["doc_id"]
print("question doc_id:", doc)
print("index exists:", (Path("indices")/doc/"page_text").exists())
print("indices:", [p.name for p in Path("indices").iterdir() if p.is_dir()][:10])

question doc_id: hyundai_wia_qa_report_5p_b413d5d7a1
index exists: True
indices: ['hyundai_wia_qa_report_5p_b413d5d7a1', 'hyundai_wia_qa_report_20p_0b67cfac3d', 'hyundai_wia_qa_report_100p_6ec8997f40', 'hyundai_wia_qa_report_50p_8e9a5d984d', 'hyundai_wia_qa_report_10p_e6a3b05064']


In [15]:
import subprocess, sys
from pathlib import Path

has = Path("models/gemma-3-4b-it-q4_0.gguf").exists()
cmd = [
    sys.executable, "scripts/run_eval_harness.py",
    "--config", "configs/experiments/eval_hw_wia_colab.yaml",
    "--datasets", "custom_5,custom_10,custom_20",  # 일단 짧게
    "--pipelines", "text,multimodal",
    "--retrievals", "page,hierarchical",
]
if not has:
    cmd.append("--dry-run")
    print("no GGUF -> dry-run")
else:
    print("GGUF found")

r = subprocess.run(cmd, capture_output=True, text=True)
print("exit:", r.returncode)
print(r.stdout[-6000:])
print(r.stderr[-6000:])

GGUF found
exit: 1
ency_ms=5674.1
query='이 분기보고서의 회사명은 무엇인가?'
-- COARSE (section) --
  [KEEP] hyundai_wia_qa_report_5p_b413d5d7a1::section::sec_sp_1 score=0.5184 pages=[3, 4] title='I.회사의 개요'
  [KEEP] hyundai_wia_qa_report_5p_b413d5d7a1::section::sec_sp_0 score=0.5101 pages=[0, 1, 2] title='분 기 보 고서'
-- FINE (page/paragraph) --
  [KEEP] hyundai_wia_qa_report_5p_b413d5d7a1::page::3 score=0.5137 pages=[3] preview='I.회사의 개요\n1.회사의 개요\n기업공시서식 작성기준에 따라 분기보고서에는 본 항목을 기재하지 않았습니다.\n2.회사의 연혁\n3.자본금 변동사항\n'
  [KEEP] hyundai_wia_qa_report_5p_b413d5d7a1::page::0 score=0.4879 pages=[0] preview='분 기 보 고서\n(제51기)\n2026년01월 01일\n부터\n사업연도\n2026년 03월 31일\n까지\n금융위원회\n한국거래소 귀중\n2026년5월14일\n'
  [KEEP] hyundai_wia_qa_report_5p_b413d5d7a1::page::4 score=0.4654 pages=[4] preview='II.사업의 내용\n1.사업의 개요\n당사는 차량부품,모빌리티솔루션 및 특수사업을 주요 사업으로 영위하고 있습니다.차량부\n품 부문은 글로벌 완성차 '
  [drop] hyundai_wia_qa_report_5p_b413d5d7a1::page::1 score=0.4436 pages=[1] preview='[대표이사 등의확인]\n대표이사등의확인·서명\n확인서\n우리는 당사의 대표이사및 신고업무담당이사로서 이 보고서의

In [16]:
import subprocess, sys
from pathlib import Path

assert Path("models/gemma-3-4b-it-q4_0.gguf").exists(), "GGUF missing"
cmd = [
    sys.executable, "scripts/run_eval_harness.py",
    "--config", "configs/experiments/eval_hw_wia_colab.yaml",
    "--datasets", "custom_5",
    "--pipelines", "text",          # multimodal은 나중에
    "--retrievals", "page",
    "--top-k", "3",
    "--limit", "2",
    "--device", "cuda",
    # --dry-run 없음
]
print(" ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print("exit:", r.returncode)
print(r.stdout[-8000:])
print(r.stderr[-8000:])

/usr/bin/python3 scripts/run_eval_harness.py --config configs/experiments/eval_hw_wia_colab.yaml --datasets custom_5 --pipelines text --retrievals page --top-k 3 --limit 2 --device cuda
exit: 0
[16:49:44] INFO pdf_vlm.index.text_embedder: Loaded text embedder BAAI/bge-m3 on cuda
[16:49:45] INFO pdf_vlm.llm.gemma: Loading Gemma 3 text-only from /content/pdf_vlm_repo/models/gemma-3-4b-it-q4_0.gguf
[16:49:47] INFO pdf_vlm.rag.text_only: qa_id=hw5_q_company retrieval=page pages=[3, 0, 4] latency_ms=ret:13656.4 gen:836.8 e2e:15931.4 answer='현대위아 주식회사'
[16:49:49] INFO pdf_vlm.rag.text_only: qa_id=hw5_q_overview retrieval=page pages=[4, 0, 1] latency_ms=ret:727.3 gen:1051.3 e2e:1779.1 answer='차량부품, 모빌리티솔루션, 특수사업'
[16:49:49] INFO pdf_vlm.eval.harness: cell custom_5__text_page_k3 -> 2 rows (correctness_mean=1.0000)
[16:49:50] INFO pdf_vlm.eval.viz: Wrote figure /content/pdf_vlm_repo/results/figures/eval_eval_hw_wia_colab_20260725_164930/length_vs_accuracy.png
[16:49:50] INFO pdf_vlm.eval.viz: W

In [17]:
import subprocess, sys
cmd = [
    sys.executable, "scripts/run_eval_harness.py",
    "--config", "configs/experiments/eval_hw_wia_colab.yaml",
    "--datasets", "custom_5",
    "--pipelines", "text,multimodal",
    "--retrievals", "page,hierarchical",
    "--device", "cuda",
]
r = subprocess.run(cmd, capture_output=True, text=True)
print("exit:", r.returncode)
print(r.stdout[-8000:]); print(r.stderr[-4000:])

exit: 1
[16:51:39] INFO pdf_vlm.index.text_embedder: Loaded text embedder BAAI/bge-m3 on cuda
[16:51:40] INFO pdf_vlm.llm.gemma: Loading Gemma 3 text-only from /content/pdf_vlm_repo/models/gemma-3-4b-it-q4_0.gguf
[16:51:42] INFO pdf_vlm.rag.text_only: qa_id=hw5_q_company retrieval=page pages=[3, 0, 4] latency_ms=ret:14083.6 gen:834.6 e2e:16496.4 answer='현대위아 주식회사'
[16:51:44] INFO pdf_vlm.rag.text_only: qa_id=hw5_q_overview retrieval=page pages=[4, 0, 1] latency_ms=ret:717.3 gen:1038.8 e2e:1756.6 answer='차량부품, 모빌리티솔루션, 특수사업'
[16:51:46] INFO pdf_vlm.rag.text_only: qa_id=hw5_q_products retrieval=page pages=[4, 0, 3] latency_ms=ret:711.6 gen:1125.6 e2e:1837.8 answer='엔진, 모듈, 등속조인트, 4WD 부품 및 열관리 관련 부품'
[16:51:48] INFO pdf_vlm.rag.text_only: qa_id=hw5_q_mobility retrieval=page pages=[4, 0, 3] latency_ms=ret:719.2 gen:1088.5 e2e:1808.2 answer='제조와 물류 현장의 자동화 ·지능화를 위한 솔루션을 제공하고 있습니다.'
[16:51:48] INFO pdf_vlm.eval.harness: cell custom_5__text_page_k3 -> 4 rows (correctness_mean=0.7500)
[16:51:5

In [18]:
import subprocess, sys, gc
try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()

def run(pipelines, datasets="custom_5"):
    cmd = [
        sys.executable, "scripts/run_eval_harness.py",
        "--config", "configs/experiments/eval_hw_wia_colab.yaml",
        "--datasets", datasets,
        "--pipelines", pipelines,
        "--retrievals", "page,hierarchical",
        "--device", "cuda",
    ]
    print(">>>", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("exit:", r.returncode)
    print(r.stdout[-6000:])
    print(r.stderr[-3000:])
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    return r.returncode

# 1) text 전부
run("text", "custom_5")

# 2) multimodal만 (text 언로드된 새 프로세스)
run("multimodal", "custom_5")

>>> /usr/bin/python3 scripts/run_eval_harness.py --config configs/experiments/eval_hw_wia_colab.yaml --datasets custom_5 --pipelines text --retrievals page,hierarchical --device cuda
exit: 0
preview='II.사업의 내용\n1.사업의 개요\n당사는 차량부품,모빌리티솔루션 및 특수사업을 주요 사업으로 영위하고 있습니다.차량부\n품 부문은 글로벌 완성차 '
  [drop] hyundai_wia_qa_report_5p_b413d5d7a1::page::1 score=0.4436 pages=[1] preview='[대표이사 등의확인]\n대표이사등의확인·서명\n확인서\n우리는 당사의 대표이사및 신고업무담당이사로서 이 보고서의 기재내용에 대해 상당\n한 주의를 다하'
  [drop] hyundai_wia_qa_report_5p_b413d5d7a1::page::2 score=0.3460 pages=[2] preview='전자공시시스템 dart.fss.or.kr\nPage 3'
FINAL pages=[3, 0, 4]
[16:55:12] INFO pdf_vlm.rag.text_only: qa_id=hw5_q_company retrieval=hierarchical pages=[3, 0, 4] latency_ms=ret:5597.6 gen:812.6 e2e:6410.8 answer='현대위아 주식회사'
[16:55:13] INFO pdf_vlm.retrieve.hierarchical: 
mode=hierarchical latency_ms=1091.6
query='당사(현대위아)의 사업의 개요에서 주요 사업 부문은 무엇인가?'
-- COARSE (section) --
  [KEEP] hyundai_wia_qa_report_5p_b413d5d7a1::section::sec_sp_0 score=0.5308 pages=[0, 1, 2] 

0